<a href="https://colab.research.google.com/github/shribr/Bricky/blob/main/Tools/torso-embeddings/Bricky-train-face-encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bricky — Face Embedding Training

End-to-end pipeline that produces the on-device face identification model.
Run cells top-to-bottom. Total time on a free Colab T4: **~3-5 hours**.

The face crop covers rows **17–35%** of the figure image — below the
hairline, above the neck — capturing printed expressions, skin tone,
glasses, and facial hair while **excluding hair/helmets/hats**.

**What you need before starting:**
1. Switch the runtime to a GPU: `Runtime > Change runtime type > T4 GPU`.
2. Nothing else. The notebook clones the repo, downloads the dataset, trains, and packages the artifacts.

**Output:** at the end you get a `face_embeddings_bundle.zip` to download. Unzip into your local repo at `Bricky/Resources/FaceEmbeddings/`, run `xcodegen generate`, rebuild — done.

**Can run in parallel with the torso notebook** — they use separate data dirs and output dirs.

## 1. Verify GPU

In [ ]:
!nvidia-smi || echo 'NO GPU — switch runtime to T4 before continuing'

## 2. Install dependencies

Colab already has torch + torchvision + Pillow. We add `coremltools` for the final export.

In [ ]:
!pip install -q coremltools numpy pillow requests

## 3. Clone the repo

Public repo, no token needed. If re-running, the old clone is removed first.

In [ ]:
import shutil, os

if os.path.exists('Bricky'):
    shutil.rmtree('Bricky')

!git clone --depth 1 https://github.com/shribr/Bricky.git
os.chdir('Bricky')
print('CWD:', os.getcwd())

## 4. Build the training dataset

Downloads ~14K face reference renders from Rebrickable CDN. Each is
cropped to the face band (17–35%) and saved as a 224×224 JPEG.

**Google Drive cache:** if a previous run already saved the dataset zip
to `MyDrive/Bricky/face_dataset.zip`, this step extracts it in
seconds. Otherwise the full download runs (~30 min) and the result is
cached for next time.

In [ ]:
import os, glob, zipfile, time

# ── Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Give Drive a moment to sync the filesystem
time.sleep(3)

DRIVE_DIR   = '/content/drive/MyDrive/Bricky'
CACHE_ZIP   = os.path.join(DRIVE_DIR, 'face_dataset.zip')
DATA_DIR    = 'Tools/torso-embeddings/data-face'
FIGURES_DIR = os.path.join(DATA_DIR, 'figures')

# Diagnostic: show what's actually in the Drive folder
print(f'Drive dir exists: {os.path.exists(DRIVE_DIR)}')
if os.path.exists(DRIVE_DIR):
    contents = os.listdir(DRIVE_DIR)
    print(f'Contents of {DRIVE_DIR}: {contents}')
else:
    print(f'{DRIVE_DIR} not found — listing MyDrive root:')
    my_drive = '/content/drive/MyDrive'
    if os.path.exists(my_drive):
        print(os.listdir(my_drive)[:30])

print(f'Looking for: {CACHE_ZIP}')
print(f'Exists: {os.path.exists(CACHE_ZIP)}')

if os.path.exists(CACHE_ZIP):
    print(f'\u2714 Found cached dataset at {CACHE_ZIP}')
    os.makedirs(DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(CACHE_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR)
    n = len(glob.glob(os.path.join(FIGURES_DIR, '*.jpg')))
    print(f'  Restored {n} face crops from Drive cache')
else:
    print('No cached dataset in Drive — building from scratch (~30 min)…')
    !python3 Tools/torso-embeddings/build-face-dataset.py --sleep 0.05

    # Save to Drive for future runs
    n = len(glob.glob(os.path.join(FIGURES_DIR, '*.jpg')))
    if n > 0:
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print(f'Caching {n} face crops to Google Drive…')
        with zipfile.ZipFile(CACHE_ZIP, 'w', zipfile.ZIP_STORED) as zf:
            for root, _dirs, files in os.walk(DATA_DIR):
                for f in files:
                    fp = os.path.join(root, f)
                    zf.write(fp, os.path.relpath(fp, DATA_DIR))
        mb = os.path.getsize(CACHE_ZIP) / (1024 * 1024)
        print(f'\u2714 Saved {CACHE_ZIP} ({mb:.1f} MB)')

### Sanity check and manifest recovery

Count images and verify the manifest exists. If the build step was interrupted after downloading images but before writing the manifest, this cell regenerates it from whatever images are on disk.

In [ ]:
import glob, json, os
from PIL import Image
from IPython.display import display

data_dir = 'Tools/torso-embeddings/data-face'
figs_dir = os.path.join(data_dir, 'figures')
manifest_path = os.path.join(data_dir, 'manifest.json')

imgs = sorted(glob.glob(os.path.join(figs_dir, '*.jpg')))
print(f'Have {len(imgs)} face crops')

# Regenerate manifest if missing (build step was interrupted)
if not os.path.exists(manifest_path) and imgs:
    entries = [{"id": os.path.splitext(os.path.basename(f))[0], "name": ""} for f in imgs]
    with open(manifest_path, 'w') as f:
        json.dump({"figures": entries}, f, separators=(",", ":"))
    print(f'Regenerated manifest with {len(entries)} entries')
elif os.path.exists(manifest_path):
    m = json.load(open(manifest_path))
    print(f'Manifest has {len(m["figures"])} entries')
else:
    print('ERROR: No images found. Re-run the build step above.')

if imgs:
    display(Image.open(imgs[len(imgs) // 2]).resize((112, 112)))

## 5. Train the encoder (~6-8 hrs on T4)

Self-supervised contrastive (SimCLR-style). Two augmented views of each
face crop are pulled together; views from different figures are pushed
apart. Defaults: 60 epochs, batch 256, ResNet18 backbone.

Every 5 epochs the script logs the average cosine similarity of random
embedding pairs — this should drop from ~0.6 toward ~0.1 as training
progresses. If it stalls above 0.4, the model isn't learning enough.

In [ ]:
CKPT_DIR = '/content/drive/MyDrive/Bricky/face_checkpoints'
!mkdir -p "$CKPT_DIR"

!python3 Tools/torso-embeddings/train-face-encoder.py \
    --epochs 60 \
    --batch-size 256 \
    --warmup-epochs 5 \
    --num-workers 2 \
    --checkpoint-dir "$CKPT_DIR" \
    --resume

## 6. Embed the full catalog (~10 min)

Runs the trained encoder over every face crop and writes the bundled vector index that ships with the iOS app. Explicit paths avoid any CWD confusion in Colab.

In [ ]:
!python3 Tools/torso-embeddings/embed-face-catalog.py \
    --checkpoint Tools/torso-embeddings/out-face/face_encoder.pt \
    --data Tools/torso-embeddings/data-face \
    --output Bricky/Resources/FaceEmbeddings

## 7. Export to CoreML (~1 min)

Converts the PyTorch encoder to CoreML for on-device inference.

> **Note:** Colab has a known `BlobWriter not loaded` bug with the `mlprogram` format in `coremltools`. We use `neuralnetwork` format instead, which works identically on iOS 17+ and avoids the bug.

In [ ]:
import sys, os
sys.path.insert(0, 'Tools/torso-embeddings')
from pathlib import Path
from importlib.machinery import SourceFileLoader

_TRAINER = SourceFileLoader(
    'train_face_encoder',
    'Tools/torso-embeddings/train-face-encoder.py'
).load_module()
FaceEncoder = _TRAINER.FaceEncoder

import torch
import torch.nn as nn
import coremltools as ct

class InferenceWrapper(nn.Module):
    """Backbone embeddings (512-D, L2-normalized).

    ImageNet normalization is baked into the graph so the CoreML
    ImageType input only needs [0,255] -> [0,1] scaling."""

    _MEAN = [0.485, 0.456, 0.406]
    _STD  = [0.229, 0.224, 0.225]

    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.register_buffer('mean', torch.tensor(self._MEAN).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor(self._STD).view(1, 3, 1, 1))

    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.encoder.encode(x)

ckpt_path = 'Tools/torso-embeddings/out-face/face_encoder.pt'
ckpt = torch.load(ckpt_path, map_location='cpu')
model = FaceEncoder(embed_dim=ckpt.get('embed_dim', 256))
model.load_state_dict(ckpt['state_dict'])
model.eval()
wrapped = InferenceWrapper(model).eval()

example = torch.randn(1, 3, 224, 224)
traced = torch.jit.trace(wrapped, example)

image_input = ct.ImageType(
    name='image',
    shape=(1, 3, 224, 224),
    scale=1.0 / 255.0,
    bias=[0.0, 0.0, 0.0],
    color_layout=ct.colorlayout.RGB,
)

mlmodel = ct.convert(
    traced,
    inputs=[image_input],
    outputs=[ct.TensorType(name='embedding')],
    convert_to='neuralnetwork',
)
mlmodel.short_description = 'Bricky face-region encoder (ResNet18, 512-D, L2-normalized)'

out_path = Path('Bricky/Resources/FaceEmbeddings/FaceEncoder.mlmodel')
out_path.parent.mkdir(parents=True, exist_ok=True)
mlmodel.save(str(out_path))
print(f'Wrote {out_path} ({out_path.stat().st_size / (1024*1024):.1f} MB)')

## 8. Package and download

Bundles all three artifacts into `face_embeddings_bundle.zip`.

In [ ]:
import os, shutil
from pathlib import Path

out_dir = Path('Bricky/Resources/FaceEmbeddings')
expected = ['face_embeddings.bin', 'face_embeddings_index.json',
            'face_embeddings_mean.bin', 'FaceEncoder.mlmodel']
missing = [e for e in expected if not (out_dir / e).exists()]
if missing:
    raise RuntimeError(f'Pipeline did not produce: {missing}')

zip_path = '/content/face_embeddings_bundle'
shutil.make_archive(zip_path, 'zip', out_dir)
size_mb = os.path.getsize(zip_path + '.zip') / (1024 * 1024)
print(f'Wrote {zip_path}.zip ({size_mb:.1f} MB)')
for e in expected:
    p = out_dir / e
    print(f'  {e}: {p.stat().st_size / (1024 * 1024):.2f} MB')

# Also save the bundle to Google Drive so it survives session disconnects.
drive_dir = '/content/drive/MyDrive/Bricky'
os.makedirs(drive_dir, exist_ok=True)
drive_zip = os.path.join(drive_dir, 'face_embeddings_bundle.zip')
shutil.copy2(zip_path + '.zip', drive_zip)
print(f'\u2714 Backed up to {drive_zip}')

In [ ]:
from google.colab import files
files.download('/content/face_embeddings_bundle.zip')

## 9. Install on your Mac

Run these commands locally after the download completes:

```bash
cd /path/to/Bricky
mkdir -p Bricky/Resources/FaceEmbeddings
unzip -o ~/Downloads/face_embeddings_bundle.zip -d Bricky/Resources/FaceEmbeddings/
xcodegen generate
xcodebuild -project Bricky.xcodeproj -scheme Bricky \
    -destination 'platform=iOS Simulator,id=YOUR_SIMULATOR_ID' \
    build
```

On the next scan, the cascade automatically uses the trained model. The runtime checks `FaceEmbeddingService.shared.isAvailable` and lights up the face embedding phase once the artifacts are present.